In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

def process_glyphs(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    RED = 15
    BLUE = 45
    GREEN = 75
    YELLOW = 105

    def smooth(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.createCLAHE(2.0, (8, 8)).apply(gray)
        gray = cv2.GaussianBlur(gray, (3, 3), 0)

        _, binary = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        inverted = cv2.bitwise_not(binary)

        kernel = np.ones((3,3), np.uint8)

        dilated = cv2.dilate(inverted, kernel, iterations=1)

        binary = cv2.bitwise_not(dilated)
        return binary

    def get_bbox(img):
        coords = np.column_stack(np.where(img < 250))
        if coords.size == 0:
            return None
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0)
        return y0, y1, x0, x1

    # classify using filename
    def classify(name):
        name = name.lower()
        if name.startswith("capital") or name[0].isdigit():
            return 1  # RED → GREEN
        if any(k in name for k in ["small_g", "small_p", "small_q", "small_y", "small_j"]):
            return 3  # BLUE → YELLOW
        return 2      # BLUE → GREEN

    for fname in os.listdir(input_folder):
        if not fname.endswith(".png"):
            continue

        img = cv2.imread(os.path.join(input_folder, fname))
        if img is None:
            continue

        binary = smooth(img)
        bbox = get_bbox(binary)
        if bbox is None:
            continue

        y0, y1, x0, x1 = bbox
        glyph = binary[y0:y1+1, x0:x1+1]

        h, w = glyph.shape
        gtype = classify(fname)

        # select vertical region
        if gtype == 1:
            top, bottom = RED, GREEN
        elif gtype == 2:
            top, bottom = BLUE, GREEN
        else:
            top, bottom = BLUE, YELLOW

        target_h = bottom - top

        # scale glyph to exactly fit region
        scale = target_h / h
        new_w = max(1, int(w * scale))
        new_h = max(1, int(h * scale))

        glyph = cv2.resize(glyph, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

        # place on fixed canvas
        canvas = np.ones((120, 140), dtype=np.uint8) * 255
        x_offset = (canvas.shape[1] - new_w) // 2
        y_offset = top

        canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = glyph
        size = 64

        h, w = canvas.shape
        scale_sq = size / max(h, w)

        new_h = int(h * scale_sq)
        new_w = int(w * scale_sq)

        resized = cv2.resize(canvas, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

        square = np.ones((size, size), dtype=np.uint8) * 255

        y_off = (size - new_h) // 2
        x_off = (size - new_w) // 2

        square[y_off:y_off+new_h, x_off:x_off+new_w] = resized

        Image.fromarray(square).save(os.path.join(output_folder, fname))

    print("Done.")

In [11]:
process_glyphs("inputs", "output")

Done.
